# 01 — EDA Aprofundada: German Credit Risk

**Camada 1 do projeto sênior.** Cobrimos:

1. Carregamento e inspeção dos dados brutos (`data/raw/credit_risk_dataset.csv`)
2. Relatório automático com **ydata-profiling**
3. Ranking de poder preditivo: **Information Value (IV)**
4. Visualização de **Weight of Evidence (WoE)** por bin
5. Contexto macroeconômico brasileiro (via `python-bcb`)
6. SHAP — explicabilidade global (requer `poetry install --with full`)

> Paleta de cores do projeto: inadimplente = `#E24B4A` · adimplente = `#639922`

In [ ]:
import sys
from pathlib import Path

# Garante que src/ está no path quando executado pelo Jupyter
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams['figure.dpi'] = 120

from src.data.make_dataset import load_raw_csv, clean_german_credit
from src.features.iv_woe import build_woe_features, iv_woe_single_feature
from src.visualization.plots import plot_iv_ranking, plot_woe_bars, COLOR_BAD, COLOR_GOOD

# Paleta
COLORS = {0: COLOR_BAD, 1: COLOR_GOOD}   # 0=mau, 1=bom

RAW_CSV = ROOT / 'data' / 'raw' / 'credit_risk_dataset.csv'
print('ROOT:', ROOT)
print('CSV existe?', RAW_CSV.exists())

## 1 · Carregamento e Perfil da Carteira

In [ ]:
df_raw = load_raw_csv(RAW_CSV)
df = clean_german_credit(df_raw)

print(f'Shape: {df.shape}')
print(f"Inadimplentes (0): {(df.credit_risk==0).sum()} | Adimplentes (1): {(df.credit_risk==1).sum()}")
print(f"Taxa de inadimplência: {(df.credit_risk==0).mean():.1%}")
df.head(3)

In [ ]:
# Distribuição do alvo — assimetria de custo: mau pagador custa 5× mais
ax = df['credit_risk'].value_counts().rename({0: 'Mau (inadimplente)', 1: 'Bom (adimplente)'})\
       .plot(kind='bar', color=[COLOR_BAD, COLOR_GOOD], figsize=(5, 3), rot=0)
ax.set_title('Distribuição da variável-alvo')
ax.set_ylabel('Contagem')
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{int(bar.get_height())}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Distribuições das principais variáveis quantitativas por classe
quant_cols = ['credit_amount', 'credit_duration', 'age']
fig, axes = plt.subplots(1, len(quant_cols), figsize=(13, 3))
for ax, col in zip(axes, quant_cols):
    for label, grp in df.groupby('credit_risk'):
        name = 'Adimplente' if label == 1 else 'Inadimplente'
        ax.hist(grp[col], bins=20, alpha=0.6, color=COLORS[label], label=name)
    ax.set_title(col)
    ax.legend(fontsize=7)
plt.suptitle('Distribuição de variáveis quantitativas por classe', y=1.02)
plt.tight_layout()
plt.show()

## 2 · Relatório automático — ydata-profiling

Substitui dezenas de `value_counts()` por um único relatório HTML interativo.
Requer `poetry install --with full`.

In [ ]:
try:
    from ydata_profiling import ProfileReport  # type: ignore[import-untyped]

    profile = ProfileReport(
        df,
        title='German Credit Risk — Relatório EDA',
        explorative=True,
        minimal=False,
    )
    output_html = ROOT / 'reports' / '01_eda_profile.html'
    output_html.parent.mkdir(parents=True, exist_ok=True)
    profile.to_file(output_html)
    print(f'Relatório gravado em {output_html}')
    profile.to_notebook_iframe()  # renderiza inline no Jupyter
except ImportError:
    print('ydata-profiling não instalado. Execute: poetry install --with full')
    # Fallback: estatísticas básicas
    display(df.describe(include='all').T)

## 3 · Information Value (IV) — Poder Preditivo das Features

IV é a métrica padrão do mercado de crédito para **seleção de variáveis**.
Implementação própria em `src/features/iv_woe.py` (sem depender de scorecardpy).

| IV | Interpretação |
|----|---------------|
| < 0.02 | Inútil |
| 0.02 – 0.1 | Fraco |
| 0.1 – 0.3 | Médio |
| 0.3 – 0.5 | Forte |
| > 0.5 | Suspeito (possível overfit) |

In [ ]:
_, iv_summary, _ = build_woe_features(df, 'credit_risk', max_bins=10)

print('Top 10 features por IV:')
display(iv_summary.head(10).style.background_gradient(cmap='Greens', subset=['iv']))

fig, ax = plt.subplots(figsize=(8, 5))
plot_iv_ranking(iv_summary, top_n=20, ax=ax)
plt.tight_layout()
plt.show()

## 4 · Weight of Evidence (WoE) — Interpretação por Bin

WoE transforma a relação bom/mau em escala log — facilita leitura de risco por bin.
Positivo = mais bons que maus (bom risco). Negativo = mais maus (alto risco).

In [ ]:
# Visualiza WoE para as 6 features de maior IV
top_features = iv_summary.head(6)['feature'].tolist()

fig, axes = plt.subplots(2, 3, figsize=(14, 6))
for ax, feat in zip(axes.flat, top_features):
    _, _, bin_stats = iv_woe_single_feature(df, feat, 'credit_risk', max_bins=10)
    plot_woe_bars(bin_stats, feat, ax=ax)

plt.suptitle('WoE por bin — top 6 features por IV', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# WoE detalhado para a feature de maior IV
top1 = iv_summary.iloc[0]['feature']
iv_val, woe_map, bin_stats = iv_woe_single_feature(df, top1, 'credit_risk', max_bins=10)

print(f"Feature: {top1} | IV = {iv_val:.4f}")
total = bin_stats['count'].sum()
bin_stats['%_total'] = (bin_stats['count'] / total * 100).round(1)
bin_stats['%_bom']   = (bin_stats['good'] / bin_stats['count'] * 100).round(1)
display(bin_stats)

## 5 · Contexto Macroeconômico Brasileiro

O GCR foi coletado na Alemanha (1970–80). A abordagem correta é enriquecer a **análise**
com o contexto macro BR atual via `python-bcb`.

**Narrativa:** *Como o perfil de risco do GCR se comportaria sob diferentes regimes macroeconômicos?
O que as variáveis universais de inadimplência nos dizem sobre o crédito no Brasil hoje?*

In [ ]:
try:
    from bcb import sgs  # type: ignore[import-untyped]

    series = {
        'selic': 432,
        'ipca': 433,
        'desemprego': 24369,
        'credito_pf': 20539,
        'inadimplencia': 21082,
    }
    df_macro = sgs.get(series, start='2018-01-01')
    df_macro.dropna(how='all', inplace=True)

    fig, axes = plt.subplots(len(df_macro.columns), 1, figsize=(12, 3 * len(df_macro.columns)), sharex=True)
    for ax, col in zip(axes, df_macro.columns):
        ax.plot(df_macro.index, df_macro[col], color='#2e86ab')
        ax.set_title(col)
        ax.grid(axis='x', alpha=0.3)

    plt.suptitle('Séries macroeconômicas BR (BACEN / SGS)', y=1.01)
    plt.tight_layout()
    plt.show()

    print('\nÚltimos valores disponíveis:')
    display(df_macro.tail(3).T.style.format('{:.2f}'))

    # Salva snapshot raw
    macro_out = ROOT / 'data' / 'raw' / 'macro_brasil' / 'macro_series.parquet'
    macro_out.parent.mkdir(parents=True, exist_ok=True)
    df_macro.to_parquet(macro_out)
    print(f'Snapshot macro gravado em {macro_out}')

except Exception as e:
    print(f'python-bcb indisponível ou sem conexão: {e}')
    print('Execute em ambiente com acesso à internet; snapshot em data/raw/macro_brasil/ se já coletado.')

### 5.1 · Diferenças estruturais BR × DE — análise qualitativa

| Dimensão | Alemanha (GCR) | Brasil |
|---|---|---|
| Informalidade | ~5% | ~40% → `employment_duration` subestimado |
| Rotatividade de emprego | Baixa | Alta → instabilidade de renda |
| Crédito consignado | Inexistente | Relevante → feature ausente no GCR |
| Cheque especial | `account_status` ≈ limitado | Produto distinto, taxas altíssimas |
| Regulação de modelos | GDPR | LGPD + BACEN Resolução 4.557 |

**Impacto analítico:** variáveis universais de inadimplência (prazo, montante, histórico) mantêm
poder preditivo; variáveis de emprego e conta corrente precisam ser recalibradas para o contexto BR.

## 6 · SHAP — Explicabilidade Global

Requer `poetry install --with full` e modelo treinado (`make train`).
A célula abaixo detecta automaticamente a disponibilidade.

In [ ]:
try:
    import shap  # type: ignore[import-untyped]
    from src.models.predict import load_artifact
    from src.features.build_features import build_feature_matrix

    bundle = load_artifact()
    pipe = bundle['pipeline']
    feature_names = bundle['feature_names']

    # WoE-encoded features
    _, woe_df, _ = build_woe_features(df, 'credit_risk', max_bins=10)
    X = woe_df[feature_names].values

    # SHAP sobre o classificador final do pipeline
    clf = pipe.named_steps['clf']
    explainer = shap.LinearExplainer(clf, X, feature_names=feature_names)
    shap_values = explainer(X)

    print('SHAP summary plot (global):')
    shap.plots.beeswarm(shap_values, max_display=15, show=True)

    print('\nMean |SHAP|:')
    mean_abs = pd.Series(
        np.abs(shap_values.values).mean(axis=0),
        index=feature_names,
    ).sort_values(ascending=False)
    display(mean_abs.head(10).to_frame('mean_|shap|'))

except ImportError:
    print('shap não instalado. Execute: poetry install --with full')
except FileNotFoundError:
    print('Modelo não encontrado. Execute: make train   (ou poetry run python -m src.models.train)')